# SAC Phase 3 — 500k adım (Colab)
Model Google Drive'a kaydedilir.

In [ ]:
# 1. Bağımlılıklar
!pip install stable-baselines3[extra] optuna gymnasium -q

In [ ]:
# 2. Drive bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Repo'yu klonla
!git clone https://github.com/isambais/SmartHome-EnergyRL.git
%cd SmartHome-EnergyRL

In [ ]:
# 4. Eğitim
import sys, os
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
from pathlib import Path
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from src.env.energy_env import SmartHomeEnergyEnv
from scripts.train.phase3_callback import Phase3MetricsCallback

DATA_PATH = Path('data/processed/aligned_dataset.csv')
MODEL_DIR = Path('models')
LOG_DIR   = Path('logs/sac_phase3')
MODEL_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

df     = pd.read_csv(DATA_PATH)
price  = df['price_tl_mwh'].values.astype('float32')
solar  = df['solar_kw'].values.astype('float32')
demand = df['demand_kw'].values.astype('float32')
print(f'Veri: {len(df)} saat')

def make_env_fn():
    return SmartHomeEnergyEnv(
        price_data=price, solar_data=solar, demand_data=demand,
        price_unit='tl_per_mwh', enable_deferrable=True,
        deferrable_load_power_kw=1.5, deferrable_load_hours=1.0,
        deferrable_window=(6, 22), deferrable_penalty_coef=2.0,
        max_activations_per_day=2,
    )

LEARNING_RATE = 0.00023688639503640813
BUFFER_SIZE   = 50_000
BATCH_SIZE    = 128
GAMMA         = 0.9857514384317185
TAU           = 0.030454635575417232
NET_ARCH_SIZE = 512

train_env = make_vec_env(make_env_fn, n_envs=1, seed=42)
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, gamma=GAMMA)

eval_env = make_vec_env(make_env_fn, n_envs=1)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)

eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=str(MODEL_DIR / 'sac_phase3_best'),
    log_path=str(LOG_DIR),
    eval_freq=5_000, n_eval_episodes=10,
    deterministic=True, verbose=1,
)
phase3_cb = Phase3MetricsCallback(verbose=0)

model = SAC(
    'MlpPolicy', train_env,
    learning_rate=LEARNING_RATE, buffer_size=BUFFER_SIZE,
    batch_size=BATCH_SIZE, gamma=GAMMA, tau=TAU,
    policy_kwargs=dict(net_arch=[NET_ARCH_SIZE, NET_ARCH_SIZE]),
    verbose=1, device='auto', tensorboard_log=str(LOG_DIR), seed=42,
)

print('SAC Phase 3 eğitimi başlıyor (500k adım)...')
model.learn(total_timesteps=500_000, callback=[eval_cb, phase3_cb], progress_bar=True)

model.save(str(MODEL_DIR / 'sac_phase3_final'))
train_env.save(str(MODEL_DIR / 'sac_phase3_vecnormalize.pkl'))
print('Eğitim tamamlandı.')

In [ ]:
# 5. Drive'a kaydet
import shutil
DRIVE_DIR = '/content/drive/MyDrive/SmartHome-EnergyRL/models'
os.makedirs(DRIVE_DIR, exist_ok=True)

files_to_save = [
    'models/sac_phase3_final.zip',
    'models/sac_phase3_vecnormalize.pkl',
    'models/sac_phase3_best/best_model.zip',
]
for f in files_to_save:
    if Path(f).exists():
        shutil.copy(f, DRIVE_DIR)
        print(f'Kaydedildi: {f} → {DRIVE_DIR}')
    else:
        print(f'UYARI: {f} bulunamadi')